# Hands-On CP-SAT: O4A Hands-On Session

There is a solver out there that keeps winning OR challenges, and not enough people know about it: **CP-SAT** from Google OR-Tools.

In this notebook we model and solve three problems from the [MiniZinc Challenge 2026](https://www.minizinc.org/challenge/2026/results/) with CP-SAT:
1. **Nonogram**, a logic puzzle: we learn the CP-SAT basics (Boolean variables, enforcement literals) and our first global constraint, `add_automaton`.
2. **Workforce allocation & scheduling**, an industrial problem: we meet *interval variables* and `add_no_overlap`, and compare CP-SAT with a MIP formulation (`pyomo` + `highspy`, as in the [first hands-on session](https://github.com/jpjj/O4A-Hands-On-Modeling)).
3. **Orthorio**, routing conveyor belts in the game *Factorio*: we use `add_circuit` to route several belts at once.

All three problems turned out to be hard for MIP solvers. Let's see why CP-SAT handles them so well.

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

This is the "homework" notebook that you can do prior to the hands-on live session. You will find these light bulbs 💡 as you progress.

They will ask you to do some tasks or reflect on some topics introduced. After progressing through this notebook, I am sure you will enjoy the live session even more!

Good luck and have fun!

</div>

## Setup (Google Colab)

Run this cell to install the required packages. If you're running locally, you can skip this cell.

In [ ]:
# Install required packages (run this cell in Google Colab)
# Note: OR-Tools ships its own copy of HiGHS. highspy 1.12.0 is compatible with it, newer versions crash when both are imported.
!pip install ortools==9.15.6755 highspy==1.12.0 pyomo pydantic matplotlib --quiet

In [ ]:
# Imports
import json
import time
import urllib.request
from itertools import combinations
from pathlib import Path
from typing import Literal, NamedTuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pyomo.environ as pyo
from ortools.sat.python import cp_model
from pydantic import BaseModel, Field, model_validator
from pyomo.contrib.appsi.solvers.highs import Highs

In [ ]:
# Utility function for loading the challenge instances
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!    You can skip this Code      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

DATA_URL = "https://raw.githubusercontent.com/jpjj/O4A-Hands-On-CP-SAT/main/data"


def load_instance(problem: str, name: str) -> dict:
    """Load an instance from the local data folder or, e.g. in Colab, from GitHub."""
    local = Path("data") / problem / f"{name}.json"
    if local.exists():
        return json.loads(local.read_text())
    with urllib.request.urlopen(f"{DATA_URL}/{problem}/{name}.json") as response:
        return json.loads(response.read())

---
# Part 0: Hello CP-SAT

CP-SAT is **declarative**: we describe *what* a solution looks like (variables, constraints, objective), and the solver figures out *how* to find it.

Every CP-SAT program follows the same four steps:
1. Create a `CpModel`.
2. Add variables: `new_bool_var`, `new_int_var`, ...
3. Add constraints and, optionally, an objective.
4. Solve with a `CpSolver` and read the values.

All variables are **integers** or **Booleans**. CP-SAT has no continuous variables. If you have fractional data, scale it, e.g. work in minutes instead of hours.

In [ ]:
model = cp_model.CpModel()

x = model.new_int_var(0, 10, "x")
y = model.new_int_var(0, 10, "y")
b = model.new_bool_var("b")

model.add(x + y <= 12)
model.add(x >= 7).only_enforce_if(b)   # if b is true, then x >= 7
model.add(y >= 7).only_enforce_if(~b)  # if b is false, then y >= 7

model.maximize(2 * x + 3 * y)

solver = cp_model.CpSolver()
status = solver.solve(model)

print(f"Status: {solver.status_name(status)}")
print(f"x = {solver.value(x)}, y = {solver.value(y)}, b = {solver.value(b)}")
print(f"Objective: {solver.objective_value}")

### `only_enforce_if`: no Big-M needed

Remember the Big-M constraints from the first hands-on session?

$$s_i \leq x_{i,j} \cdot \alpha_j + (1 - x_{i,j}) \cdot M$$

In CP-SAT we simply write `model.add(s[i] <= alpha[j]).only_enforce_if(x[i, j])`. The constraint is only active if the literal is true. There is no $M$ to choose, and the solver reasons about the implication directly.

---
# Part 1: Nonogram

## Problem Description

A nonogram is a grid puzzle. Every cell is either **filled** or **empty**.

- Every row and every column has a **clue**: a list of block lengths, e.g. `[2, 1]`.
- The clue says which blocks of consecutive filled cells appear in that line, in this order.
- Blocks are separated by at least one empty cell.

We have to find a grid that satisfies all clues. There is no objective: this is a pure **satisfaction problem**.

Nonograms were one of the 20 problems of the MiniZinc Challenge 2026. CP-SAT solved all five instances. On the four harder ones (25x25 and 35x35), Gurobi, CPLEX and HiGHS solved at most one within 20 minutes.

## Domain Model

As in the first session, we use `pydantic` to define and validate our data.

In [ ]:
# Define Problem and Solution data structures

class Nonogram(BaseModel):
    """A nonogram puzzle: one clue (list of block lengths) per row and per column."""
    name: str = ""
    rows: list[list[int]] = Field(description="clues of the rows, top to bottom")
    cols: list[list[int]] = Field(description="clues of the columns, left to right")

    @property
    def height(self) -> int:
        return len(self.rows)

    @property
    def width(self) -> int:
        return len(self.cols)

    @model_validator(mode="after")
    def check_clues(self) -> "Nonogram":
        for clue in self.rows:
            if sum(clue) + len(clue) - 1 > self.width:
                raise ValueError(f"Row clue {clue} does not fit into width {self.width}")
        for clue in self.cols:
            if sum(clue) + len(clue) - 1 > self.height:
                raise ValueError(f"Column clue {clue} does not fit into height {self.height}")
        if sum(map(sum, self.rows)) != sum(map(sum, self.cols)):
            raise ValueError("Rows and columns disagree on the number of filled cells")
        return self


def blocks_of(line: list[int]) -> list[int]:
    """Block lengths of a line of 0/1 values, e.g. [1, 1, 0, 1] -> [2, 1]."""
    blocks, run = [], 0
    for value in list(line) + [0]:
        if value:
            run += 1
        elif run:
            blocks.append(run)
            run = 0
    return blocks


class NonogramSolution(BaseModel):
    """A filled grid that satisfies all clues of its puzzle."""
    puzzle: Nonogram
    grid: list[list[int]]

    @model_validator(mode="after")
    def check_clues(self) -> "NonogramSolution":
        for r, clue in enumerate(self.puzzle.rows):
            if blocks_of(self.grid[r]) != clue:
                raise ValueError(f"Row {r} violates its clue {clue}")
        for c, clue in enumerate(self.puzzle.cols):
            if blocks_of([row[c] for row in self.grid]) != clue:
                raise ValueError(f"Column {c} violates its clue {clue}")
        return self

In [ ]:
# Define a plotting function for nonograms
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!    You can skip this Code      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

def plot_nonogram(puzzle: Nonogram, solution: NonogramSolution | None = None, title: str | None = None):
    """Plot the puzzle (with clues for small puzzles) and, if given, its solution."""
    H, W = puzzle.height, puzzle.width
    show_clues = max(H, W) <= 15
    row_space = 0.6 * max(len(clue) for clue in puzzle.rows) + 0.5 if show_clues else 0
    col_space = 0.5 * max(len(clue) for clue in puzzle.cols) + 0.5 if show_clues else 0
    scale = 0.4 if show_clues else 0.2
    fig, ax = plt.subplots(figsize=((W + row_space) * scale + 0.5, (H + col_space) * scale + 0.5))

    for r in range(H):
        for c in range(W):
            filled = solution is not None and solution.grid[r][c] == 1
            ax.add_patch(mpatches.Rectangle(
                (c, H - 1 - r), 1, 1, facecolor="steelblue" if filled else "white",
                edgecolor="lightgrey", linewidth=0.5,
            ))
    for i in range(0, W + 1, 5):
        ax.plot([i, i], [0, H], color="grey", linewidth=1)
    for i in range(0, H + 1, 5):
        ax.plot([0, W], [H - i, H - i], color="grey", linewidth=1)

    if show_clues:
        for r, clue in enumerate(puzzle.rows):
            ax.text(-0.3, H - 0.5 - r, "  ".join(map(str, clue)) or "0", ha="right", va="center", fontsize=9)
        for c, clue in enumerate(puzzle.cols):
            ax.text(c + 0.5, H + 0.2, "\n".join(map(str, clue)) or "0", ha="center", va="bottom", fontsize=9)

    ax.set_xlim(-row_space, W + 0.1)
    ax.set_ylim(-0.1, H + col_space)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()

## Baby Example

Let us start with a small 10x10 puzzle. Can you solve it by hand?

In [ ]:
# Baby example: a 10x10 puzzle
baby_puzzle = Nonogram(
    name="baby",
    rows=[[4], [6], [3, 3], [2, 2, 2], [3, 3], [2, 2], [4], [4], [], [4]],
    cols=[[], [3], [5], [3, 4, 1], [2, 1, 2, 1], [2, 1, 2, 1], [3, 4, 1], [5], [3], []],
)
plot_nonogram(baby_puzzle, title="Baby Example")

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

Hello! Thank you for doing the homework prior to the hands-on session!

Before you continue, think about how you would model a single line (row or column) of a nonogram:
- What are your variables?
- How do you make sure the blocks appear in the right order and with the right lengths?
- How would you do this in a MIP? And would you do anything differently in CP?

When you feel like you engaged enough with this matter, feel free to continue reading!

</div>

## Model 1: Blocks with Enforcement Literals

The grid has one Boolean variable per cell. Every line (row or column) gets its own block variables:

| Variable | Domain | Description |
|----------|--------|-------------|
| $cell_{r,c}$ | $\{0,1\}$ | cell $(r, c)$ is filled (shared by its row and its column) |
| $start_i$ | $\{0, \dots, n - \ell_i\}$ | first cell of block $i$ of the line, $\ell_i$ is its length |
| $covers_{i,c}$ | $\{0,1\}$ | block $i$ covers cell $c$ of the line |

Constraints per line:
1. Blocks are ordered and separated by at least one empty cell: $start_i + \ell_i + 1 \leq start_{i+1}$
2. Block $i$ covers exactly $\ell_i$ cells: $\sum_c covers_{i,c} = \ell_i$
3. If block $i$ covers cell $c$, then $c$ lies inside the block: $covers_{i,c} \Rightarrow start_i \leq c \leq start_i + \ell_i - 1$
4. A cell is filled if and only if a block covers it: $cell_c = \sum_i covers_{i,c}$

Constraint 3 is where `only_enforce_if` shines.

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

Now, it is your time to code!

Below you find the implementation of Model 1. We use [OR-Tools' CP-SAT](https://developers.google.com/optimization/cp/cp_solver) via its Python API.

The code misses constraint 3.

**What you need to do:**
- Look again at the definition of the model above.
- Take a look at how the other constraints have been implemented and add constraint 3 using `only_enforce_if`.
- When you are done, run the next cells and solve the baby example. If everything goes according to plan, you should see the following output (and a nice picture):
>```
>baby (add_line_blocks): OPTIMAL after 0.01s
>```
The time may differ, of course. For satisfaction problems, CP-SAT reports `OPTIMAL` as soon as it has found a solution.

</div>

In [ ]:
def add_line_blocks(model: cp_model.CpModel, cells: list[cp_model.IntVar], clue: list[int]):
    """Model 1: one start variable per block and 'block i covers cell c' literals."""
    n = len(cells)
    starts = [model.new_int_var(0, n - length, f"start_{i}") for i, length in enumerate(clue)]
    covers = [[model.new_bool_var(f"covers_{i}_{c}") for c in range(n)] for i in range(len(clue))]

    # 1. blocks are ordered and separated by at least one empty cell
    for i in range(len(clue) - 1):
        model.add(starts[i] + clue[i] + 1 <= starts[i + 1])

    for i, length in enumerate(clue):
        # 2. block i covers exactly 'length' cells ...
        model.add(sum(covers[i]) == length)
        # 3. ... and all of them lie inside [start, start + length - 1]
        for c in range(n):
            # GAP HERE
            # if covers[i][c] is true, then starts[i] <= c and starts[i] + length - 1 >= c
            pass

    # 4. a cell is filled if and only if one block covers it
    for c in range(n):
        model.add(cells[c] == sum(covers[i][c] for i in range(len(clue))))


def create_nonogram_model(puzzle: Nonogram, add_line=add_line_blocks):
    """Create the model: one Boolean per cell and one line constraint per row and column."""
    model = cp_model.CpModel()
    grid = [[model.new_bool_var(f"cell_{r}_{c}") for c in range(puzzle.width)] for r in range(puzzle.height)]
    for r, clue in enumerate(puzzle.rows):
        add_line(model, grid[r], clue)
    for c, clue in enumerate(puzzle.cols):
        add_line(model, [grid[r][c] for r in range(puzzle.height)], clue)
    return model, grid

In [ ]:
def solve_nonogram(puzzle: Nonogram, add_line=add_line_blocks, time_limit: float = 60,
                   verbose: bool = False) -> NonogramSolution | None:
    """Solve the puzzle and return a validated solution (or None)."""
    model, grid = create_nonogram_model(puzzle, add_line)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.log_search_progress = verbose
    status = solver.solve(model)
    print(f"{puzzle.name} ({add_line.__name__}): {solver.status_name(status)} after {solver.wall_time:.2f}s")
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return NonogramSolution(puzzle=puzzle, grid=[[solver.value(cell) for cell in row] for row in grid])

# This is where you test your homework! Run the next cell.

In [ ]:
baby_solution = solve_nonogram(baby_puzzle)
plot_nonogram(baby_puzzle, baby_solution, title="Baby Example: Solution")

## A Challenge Instance (25x25)

The challenge instances are random patterns rather than pictures, and they are much bigger.

In [ ]:
medium_puzzle = Nonogram(**load_instance("nonogram", "medium_05"))
medium_solution = solve_nonogram(medium_puzzle)
plot_nonogram(medium_puzzle, medium_solution, title="medium_05 (25x25)")

## Model 2: Use a Global Constraint

Model 1 works, but it needs many helper variables, and the solver has to rediscover the structure of a line from many small constraints.

**Global constraints** capture a common substructure in a single constraint, and the solver brings a specialized propagator for it. A nonogram line is a perfect fit for `add_automaton`: a line is valid if and only if a small finite automaton accepts it.

Example: the clue `[2, 1]` corresponds to the shortest valid line `1 1 0 1`. Additional empty cells may appear before the first block, inside the gaps and after the last block:

```
 ↻0                                ↻0           ↻0
(0) ──1──▶ (1) ──1──▶ (2) ──0──▶ (3) ──1──▶ ((4))
start                                        accept
```

The state counts how many symbols of the shortest valid line we have matched so far.

In [ ]:
def line_automaton(clue: list[int]) -> tuple[int, list[int], list[tuple[int, int, int]]]:
    """Build the automaton of a clue: (start state, final states, transitions)."""
    # the shortest valid line, e.g. [2, 1] -> 1 1 0 1
    pattern = []
    for i, length in enumerate(clue):
        pattern += [1] * length + ([0] if i < len(clue) - 1 else [])

    transitions = []  # (state, value, next state)
    for state in range(len(pattern) + 1):
        in_gap = state == 0 or state == len(pattern) or pattern[state - 1] == 0
        if in_gap:
            transitions.append((state, 0, state))  # more empty cells are fine here
        if state < len(pattern):
            transitions.append((state, pattern[state], state + 1))  # match the next symbol
    return 0, [len(pattern)], transitions


def add_line_automaton(model: cp_model.CpModel, cells: list[cp_model.IntVar], clue: list[int]):
    """Model 2: the line must be accepted by the automaton of its clue."""
    start, finals, transitions = line_automaton(clue)
    model.add_automaton(cells, start, finals, transitions)


line_automaton([2, 1])

## Comparing Both Models

In [ ]:
for name in ["easy_01", "medium_05", "hard_02"]:
    puzzle = Nonogram(**load_instance("nonogram", name))
    for add_line in [add_line_blocks, add_line_automaton]:
        solve_nonogram(puzzle, add_line, time_limit=60)

In [ ]:
hard_puzzle = Nonogram(**load_instance("nonogram", "hard_03"))
hard_solution = solve_nonogram(hard_puzzle, add_line_automaton)
plot_nonogram(hard_puzzle, hard_solution, title="hard_03 (35x35)")

### Takeaways

- Without an objective, a MIP solver gets no guidance from its LP relaxation. CP-SAT's propagation and clause learning thrive on such logical problems.
- **Use global constraints** whenever your structure matches one: `add_automaton`, `add_all_different`, `add_element`, `add_allowed_assignments` (table), `add_no_overlap`, `add_cumulative`, `add_circuit`, ...

---
# Part 2: Workforce Allocation & Scheduling

## Problem Description

This problem comes from industry (Bleukx et al., *Modeling and explaining an industrial workforce allocation and scheduling problem*, CP 2025; [code](https://github.com/ML-KULeuven/Explainable-Workforce-Scheduling)) and was part of the MiniZinc Challenge 2026. It is a big brother of the worker/task problem from our first session.

There are $N$ **teams** and $K$ **tasks**.

- Every task $j$ has a duration $d_j$ and a time window: it may start at its release time $r_j$ at the earliest and must end by its due date $\delta_j$. **We decide when exactly.**
- A task can only be done by a **compatible** team.
- A team does one task at a time and cannot work during its **unavailabilities** (calendar).
- **Precedences:** some tasks must be finished before others start.
- **Same team groups:** some tasks must be done by the same team.

**Objectives** (lexicographic):
1. Minimize the number of teams used.
2. Minimize the unfairness: the difference between the maximum and the minimum working time of the used teams.

We combine both into one objective: $\left(\sum_j d_j\right) \cdot \#\text{teams} + (\max \text{worked} - \min \text{worked})$. The unfairness is always smaller than $\sum_j d_j$, so saving a team is always worth more.

## Domain Model

In [ ]:
# Define Problem data structures

class Task(BaseModel, frozen=True):
    """A task with a duration and a time window [release, due]."""
    id: int
    duration: int = Field(gt=0)
    release: int = Field(ge=0)
    due: int = Field(ge=0)
    compatible_teams: tuple[int, ...] = Field(description="teams that can do this task")

    @model_validator(mode="after")
    def check_window(self) -> "Task":
        if self.release + self.duration > self.due:
            raise ValueError(f"Task {self.id} does not fit into its time window")
        return self


class Unavailability(BaseModel, frozen=True):
    """A period in which a team cannot work."""
    team: int
    start: int = Field(ge=0)
    duration: int = Field(gt=0)


class WorkforceProblem(BaseModel):
    """Represents the problem instance."""
    name: str = ""
    horizon: int
    num_teams: int = Field(ge=1)
    tasks: list[Task]
    same_team_groups: list[list[int]] = Field(default=[], description="tasks that must be done by the same team")
    precedences: list[tuple[int, int]] = Field(default=[], description="(a, b): a must end before b starts")
    unavailabilities: list[Unavailability] = []

    @property
    def teams(self) -> range:
        return range(self.num_teams)

    @model_validator(mode="after")
    def check_references(self) -> "WorkforceProblem":
        ids = {t.id for t in self.tasks}
        for task in self.tasks:
            if not set(task.compatible_teams) <= set(self.teams):
                raise ValueError(f"Task {task.id} references an unknown team")
        for a, b in self.precedences:
            if a not in ids or b not in ids:
                raise ValueError(f"Precedence ({a}, {b}) references an unknown task")
        for group in self.same_team_groups:
            if not set(group) <= ids:
                raise ValueError(f"Group {group} references an unknown task")
        return self

In [ ]:
# Define Solution and helper functions
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!    You can skip this Code      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


class ScheduledTask(BaseModel, frozen=True):
    """A task together with its team and start time."""
    task: Task
    team: int
    start: int

    @property
    def end(self) -> int:
        return self.start + self.task.duration


class Schedule(BaseModel):
    """A feasible solution: which team does which task when."""
    problem: WorkforceProblem
    tasks: list[ScheduledTask]

    @property
    def worked(self) -> dict[int, int]:
        """Working time per used team."""
        worked = {}
        for st in self.tasks:
            worked[st.team] = worked.get(st.team, 0) + st.task.duration
        return worked

    @property
    def cost(self) -> int:
        total = sum(t.duration for t in self.problem.tasks)
        worked = self.worked.values()
        return total * len(worked) + max(worked) - min(worked)

    @model_validator(mode="after")
    def check_feasible(self) -> "Schedule":
        p = self.problem
        if sorted(st.task.id for st in self.tasks) != sorted(t.id for t in p.tasks):
            raise ValueError("Every task must be scheduled exactly once")
        team_of = {st.task.id: st.team for st in self.tasks}
        start_of = {st.task.id: st.start for st in self.tasks}
        end_of = {st.task.id: st.end for st in self.tasks}
        for st in self.tasks:
            if st.team not in st.task.compatible_teams:
                raise ValueError(f"Task {st.task.id} is assigned to an incompatible team")
            if st.start < st.task.release or st.end > st.task.due:
                raise ValueError(f"Task {st.task.id} violates its time window")
        for a, b in p.precedences:
            if end_of[a] > start_of[b]:
                raise ValueError(f"Precedence {a} -> {b} is violated")
        for group in p.same_team_groups:
            if len({team_of[t] for t in group}) > 1:
                raise ValueError(f"Group {group} is split over several teams")
        for k in p.teams:
            busy = sorted([(st.start, st.end) for st in self.tasks if st.team == k]
                          + [(u.start, u.start + u.duration) for u in p.unavailabilities if u.team == k])
            for (_, end), (start, _) in zip(busy, busy[1:]):
                if start < end:
                    raise ValueError(f"Team {k} is busy twice at time {start}")
        return self


def plot_schedule(schedule: Schedule, title: str | None = None):
    """Plot a Gantt chart with one row per used team."""
    used = sorted(schedule.worked)
    fig, ax = plt.subplots(figsize=(14, max(2, 0.6 * len(used) + 1)))
    for row, team in enumerate(used):
        for u in schedule.problem.unavailabilities:
            if u.team == team:
                ax.barh(row, u.duration, left=u.start, height=0.5, color="grey", alpha=0.3,
                        hatch="//", edgecolor="grey")
        for st in schedule.tasks:
            if st.team == team:
                ax.barh(row, st.task.duration, left=st.start, height=0.5, color="steelblue",
                        edgecolor="darkblue", linewidth=1)
                ax.text(st.start + st.task.duration / 2, row, str(st.task.id), ha="center", va="center",
                        color="white", fontweight="bold", fontsize=8)
    ax.set_yticks(range(len(used)))
    ax.set_yticklabels([f"Team {k} ({schedule.worked[k]})" for k in used])
    ax.invert_yaxis()
    ax.set_xlim(0, schedule.problem.horizon)
    ax.set_xlabel("Time")
    ax.set_title(title or f"{schedule.problem.name}: {len(used)} teams, cost {schedule.cost}")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Baby Example

Three teams, six tasks and a working day of 8 hours. Team 0 has a lunch break from 4:00 to 5:00.

In [ ]:
baby_problem = WorkforceProblem(
    name="baby",
    horizon=8,
    num_teams=3,
    tasks=[
        Task(id=0, duration=2, release=0, due=4, compatible_teams=(0, 1, 2)),
        Task(id=1, duration=3, release=0, due=8, compatible_teams=(0, 1)),
        Task(id=2, duration=2, release=2, due=6, compatible_teams=(0, 1, 2)),
        Task(id=3, duration=1, release=5, due=8, compatible_teams=(1, 2)),
        Task(id=4, duration=2, release=0, due=8, compatible_teams=(0, 1, 2)),
        Task(id=5, duration=2, release=4, due=8, compatible_teams=(0, 2)),
    ],
    precedences=[(0, 2)],          # task 0 before task 2
    same_team_groups=[[3, 5]],     # tasks 3 and 5 are done by the same team
    unavailabilities=[Unavailability(team=0, start=4, duration=1)],
)
print(f"{len(baby_problem.tasks)} tasks, {baby_problem.num_teams} teams, "
      f"total duration {sum(t.duration for t in baby_problem.tasks)}h")

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

Before you continue, think about how you would model this problem as a MIP.

Think about:
- How do you model that a team does at most one task at a time? (Remember the first session.)
- How do you model the time windows and the unavailabilities?
- How many variables and constraints do you need, and which of them use a Big-M?

When you feel like you engaged enough with this matter, feel free to continue reading!

</div>

## Modeling with Interval Variables

CP-SAT has a dedicated variable type for scheduling: the **interval variable**. An interval consists of
- a start $s$, a size $d$ and an end $e$ with $s + d = e$,
- and optionally a **presence literal**: an *optional interval* only exists if its literal is true.

That is exactly what we need:

| Variable | Domain | Description |
|----------|--------|-------------|
| $x_{j,k}$ | $\{0,1\}$ | team $k$ does task $j$ (only for compatible pairs) |
| $start_j$ | $\{r_j, \dots, \delta_j - d_j\}$ | start time of task $j$ |
| $interval_{j,k}$ | optional interval | $[start_j, start_j + d_j)$, present iff $x_{j,k}$ |
| $used_k$ | $\{0,1\}$ | team $k$ does at least one task |

The heart of the model is one global constraint per team:

`model.add_no_overlap(intervals of team k + unavailabilities of team k)`

Only present intervals take part. No pairwise ordering variables, no Big-M!

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

It is again time to fill the gaps!

The model below misses its most important part: the intervals and the no-overlap constraints.

**What you need to do:**
- Create one optional interval per compatible (task, team) pair with `model.new_optional_fixed_size_interval_var(start, size, is_present, name)`.
- For every team, add one `model.add_no_overlap(...)` over its task intervals and its unavailabilities. For the unavailabilities, use fixed intervals: `model.new_fixed_size_interval_var(start, size, name)`.
- You might want to take a look at the [CP-SAT documentation on scheduling](https://developers.google.com/optimization/scheduling/job_shop).

When you are done, run the next cells. For the baby example you should see:
>```
>baby: OPTIMAL after 0.01s, cost 26, bound 26
>```

</div>

In [ ]:
def add_team_symmetry_breaking(model: cp_model.CpModel, problem: WorkforceProblem, used, worked):
    """Teams with the same compatible tasks and the same calendar are interchangeable.
    Use them in order and sort them by working time."""
    groups = {}
    for k in problem.teams:
        tasks_k = tuple(t.id for t in problem.tasks if k in t.compatible_teams)
        calendar_k = tuple((u.start, u.duration) for u in problem.unavailabilities if u.team == k)
        groups.setdefault((tasks_k, calendar_k), []).append(k)
    for group in groups.values():
        for k1, k2 in zip(group, group[1:]):
            model.add(used[k1] >= used[k2])
            model.add(worked[k1] >= worked[k2])


def create_workforce_model(problem: WorkforceProblem, symmetry_breaking: bool = True):
    """Create the CP-SAT model for workforce allocation & scheduling."""
    model = cp_model.CpModel()
    tasks, teams = problem.tasks, problem.teams
    total = sum(t.duration for t in tasks)

    # Variables
    x = {(t.id, k): model.new_bool_var(f"x_{t.id}_{k}") for t in tasks for k in t.compatible_teams}
    start = {t.id: model.new_int_var(t.release, t.due - t.duration, f"start_{t.id}") for t in tasks}
    used = [model.new_bool_var(f"used_{k}") for k in teams]

    # Every task is done by exactly one compatible team
    for t in tasks:
        model.add_exactly_one(x[t.id, k] for k in t.compatible_teams)

    # A team does one task at a time and respects its calendar
    # GAP HERE
    # 1. create one optional interval per compatible (task, team) pair
    # 2. for every team: add_no_overlap over its task intervals and its unavailabilities

    # Precedences: task a ends before task b starts
    duration = {t.id: t.duration for t in tasks}
    for a, b in problem.precedences:
        model.add(start[a] + duration[a] <= start[b])

    # Tasks of a group are done by the same team
    for first, *others in problem.same_team_groups:
        for other in others:
            for k in teams:
                if (first, k) in x or (other, k) in x:
                    model.add(x.get((first, k), 0) == x.get((other, k), 0))

    # A team is used if and only if it does at least one task
    for k in teams:
        assigned = [x[t.id, k] for t in tasks if (t.id, k) in x]
        for lit in assigned:
            model.add_implication(lit, used[k])
        model.add_bool_or(assigned).only_enforce_if(used[k])

    # Fairness: maximum and minimum working time of the used teams
    worked = [sum(t.duration * x[t.id, k] for t in tasks if (t.id, k) in x) for k in teams]
    max_worked = model.new_int_var(0, total, "max_worked")
    min_worked = model.new_int_var(0, total, "min_worked")
    for k in teams:
        model.add(max_worked >= worked[k])
        model.add(min_worked <= worked[k]).only_enforce_if(used[k])

    if symmetry_breaking:
        add_team_symmetry_breaking(model, problem, used, worked)

    model.minimize(total * sum(used) + max_worked - min_worked)
    return model, x, start

In [ ]:
def solve_workforce(problem: WorkforceProblem, time_limit: float = 60, verbose: bool = False,
                    **kwargs) -> Schedule | None:
    """Solve the problem and return a validated schedule (or None)."""
    model, x, start = create_workforce_model(problem, **kwargs)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.log_search_progress = verbose
    status = solver.solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"{problem.name}: {solver.status_name(status)} after {solver.wall_time:.2f}s")
        return None
    print(f"{problem.name}: {solver.status_name(status)} after {solver.wall_time:.2f}s, "
          f"cost {solver.objective_value:.0f}, bound {solver.best_objective_bound:.0f}")
    task_by_id = {t.id: t for t in problem.tasks}
    scheduled = [
        ScheduledTask(task=task_by_id[j], team=k, start=solver.value(start[j]))
        for (j, k), lit in x.items()
        if solver.boolean_value(lit)
    ]
    return Schedule(problem=problem, tasks=scheduled)

# This is where you test your homework! Run the next cell.

In [ ]:
baby_schedule = solve_workforce(baby_problem)
plot_schedule(baby_schedule)

## A Challenge Instance (58 tasks, 18 teams)

In [ ]:
problem_71 = WorkforceProblem(**load_instance("workforce", "instance_71"))
print(f"{len(problem_71.tasks)} tasks, {problem_71.num_teams} teams, "
      f"{len(problem_71.precedences)} precedences, {len(problem_71.same_team_groups)} same team groups")

schedule_71 = solve_workforce(problem_71)
plot_schedule(schedule_71)

## Best Practice: Break Symmetries

Most teams in this instance are interchangeable: they can do the same tasks and have the same calendar. A solution with 5 teams can be relabeled in $18 \cdot 17 \cdot 16 \cdot 15 \cdot 14 > 1{,}000{,}000$ ways, and the solver has to rule out every one of them to prove optimality.

`add_team_symmetry_breaking` tells the solver to use interchangeable teams in a fixed order. Let's switch it off and compare. Without it, proving optimality takes much longer: with the two cores of a free Colab instance, CP-SAT does not even manage it within 20 seconds.

In [ ]:
for symmetry_breaking in [False, True]:
    print(f"symmetry_breaking={symmetry_breaking}")
    solve_workforce(problem_71, time_limit=20, symmetry_breaking=symmetry_breaking)

## Comparison: a MIP Formulation (Pyomo + HiGHS)

How would we model this as a MIP? Without interval variables, we need for every pair of tasks $(a, b)$ whose time windows overlap:
- $same_{a,b}$: are $a$ and $b$ done by the same team? ($same_{a,b} \geq x_{a,k} + x_{b,k} - 1$ for every team $k$)
- $before_{a,b}$: is $a$ done before $b$?
- two Big-M constraints:

$$s_a + d_a \leq s_b + M (1 - before_{a,b}) + M (1 - same_{a,b})$$
$$s_b + d_b \leq s_a + M \cdot before_{a,b} + M (1 - same_{a,b})$$

We apply everything we learned in the first session: a tight, pair-specific $M$, variable bounds, and the same symmetry breaking. For brevity, the MIP ignores unavailabilities; the challenge instances do not have any.

In [ ]:
def create_mip_model(problem: WorkforceProblem) -> pyo.ConcreteModel:
    """Big-M MIP formulation of the workforce problem (without unavailabilities)."""
    tasks = {t.id: t for t in problem.tasks}
    total = sum(t.duration for t in problem.tasks)
    compatible = [(t.id, k) for t in problem.tasks for k in t.compatible_teams]
    # only pairs with overlapping time windows need an ordering decision
    pairs = [(a, b) for a, b in combinations(tasks, 2)
             if tasks[a].release < tasks[b].due and tasks[b].release < tasks[a].due]

    m = pyo.ConcreteModel("Workforce - Big-M MIP")
    m.x = pyo.Var(compatible, domain=pyo.Binary)
    m.s = pyo.Var(list(tasks), bounds=lambda m, j: (tasks[j].release, tasks[j].due - tasks[j].duration))
    m.used = pyo.Var(problem.teams, domain=pyo.Binary)
    m.before = pyo.Var(pairs, domain=pyo.Binary)
    m.same = pyo.Var(pairs, bounds=(0, 1))
    m.max_worked = pyo.Var(bounds=(0, total))
    m.min_worked = pyo.Var(bounds=(0, total))

    @m.Constraint(list(tasks))
    def assign_once(m, j):
        return pyo.quicksum(m.x[j, k] for k in tasks[j].compatible_teams) == 1

    @m.Constraint(pairs, problem.teams)
    def detect_same_team(m, a, b, k):
        if (a, k) not in m.x or (b, k) not in m.x:
            return pyo.Constraint.Skip
        return m.same[a, b] >= m.x[a, k] + m.x[b, k] - 1

    @m.Constraint(pairs)
    def a_before_b(m, a, b):
        M = tasks[a].due - tasks[b].release
        return m.s[a] + tasks[a].duration <= m.s[b] + M * (1 - m.before[a, b]) + M * (1 - m.same[a, b])

    @m.Constraint(pairs)
    def b_before_a(m, a, b):
        M = tasks[b].due - tasks[a].release
        return m.s[b] + tasks[b].duration <= m.s[a] + M * m.before[a, b] + M * (1 - m.same[a, b])

    @m.Constraint(range(len(problem.precedences)))
    def precedence(m, i):
        a, b = problem.precedences[i]
        return m.s[a] + tasks[a].duration <= m.s[b]

    m.same_team = pyo.ConstraintList()
    for first, *others in problem.same_team_groups:
        for other in others:
            for k in problem.teams:
                if (first, k) in m.x or (other, k) in m.x:
                    m.same_team.add((m.x[first, k] if (first, k) in m.x else 0)
                                    == (m.x[other, k] if (other, k) in m.x else 0))

    @m.Constraint(compatible)
    def team_used(m, j, k):
        return m.x[j, k] <= m.used[k]

    def worked(k):
        return pyo.quicksum(tasks[j].duration * m.x[j, kk] for (j, kk) in compatible if kk == k)

    @m.Constraint(problem.teams)
    def max_definition(m, k):
        return m.max_worked >= worked(k)

    @m.Constraint(problem.teams)
    def min_definition(m, k):
        return m.min_worked <= worked(k) + total * (1 - m.used[k])

    # same symmetry breaking as in the CP-SAT model
    m.symmetry = pyo.ConstraintList()
    groups = {}
    for k in problem.teams:
        groups.setdefault(tuple(j for (j, kk) in compatible if kk == k), []).append(k)
    for group in groups.values():
        for k1, k2 in zip(group, group[1:]):
            m.symmetry.add(m.used[k1] >= m.used[k2])
            m.symmetry.add(worked(k1) >= worked(k2))

    m.total_costs = pyo.Objective(
        expr=total * pyo.quicksum(m.used[k] for k in problem.teams) + m.max_worked - m.min_worked,
        sense=pyo.minimize,
    )
    return m


def solve_mip(problem: WorkforceProblem, time_limit: float = 30, verbose: bool = False) -> dict:
    """Solve the MIP with HiGHS and return objective, bound and runtime."""
    start_time = time.time()
    m = create_mip_model(problem)
    solver = Highs()
    solver.config.stream_solver = verbose
    solver.config.load_solution = False
    solver.highs_options["time_limit"] = time_limit
    results = solver.solve(m)
    return {
        "status": results.termination_condition.name,
        "objective": results.best_feasible_objective,
        "bound": results.best_objective_bound,
        "time": time.time() - start_time,
    }

In [ ]:
# CP-SAT vs. MIP on the challenge instances (time limit 30s each)
rows = []
for name in ["instance_202", "instance_24_56", "instance_71", "instance_19"]:
    problem = WorkforceProblem(**load_instance("workforce", name))
    start_time = time.time()
    schedule = solve_workforce(problem, time_limit=30)
    cp_time = time.time() - start_time
    mip = solve_mip(problem, time_limit=30)
    rows.append((name, len(problem.tasks), schedule.cost if schedule else None, cp_time,
                 mip["objective"], mip["bound"], mip["time"]))

print(f"\n{'instance':16} {'tasks':>5} | {'CP-SAT':>8} {'time':>6} | {'MIP':>8} {'bound':>8} {'time':>6}")
for name, k, cp_cost, cp_time, mip_obj, mip_bound, mip_time in rows:
    fmt = lambda v: f"{v:8.0f}" if v is not None else f"{'-':>8}"
    print(f"{name:16} {k:5} | {fmt(cp_cost)} {cp_time:5.1f}s | {fmt(mip_obj)} {fmt(mip_bound)} {mip_time:5.1f}s")

### Takeaways

- Interval variables and `add_no_overlap` replace $O(K^2)$ ordering variables and Big-M constraints.
- Optional intervals combine *assignment* (which team?) and *scheduling* (when?) in one variable.
- **Break symmetries.** Interchangeable resources blow up the search space, for CP-SAT and for MIP solvers alike.
- MIP is not hopeless: on small instances HiGHS keeps up. The disjunctive structure at scale is what hurts.

---
# Part 3: Orthorio: Conveyor Belts in Factorio

## Problem Description

In the game [Factorio](https://www.factorio.com) you build factories, and items travel on **conveyor belts**. A classic puzzle: route several belts through a small area without mixing them. Julia Dijkstra submitted this problem to the MiniZinc Challenge 2026 as a hobby project called *Orthorio*.

We have a grid and $A$ belts. Every belt has a **start** (input) and an **end** (output) cell, both with a fixed direction.

- Every cell holds at most one belt piece.
- A **surface belt** moves items to the neighboring cell in its direction. The next piece may go straight or turn left or right, but not back.
- An **underground belt** dives at an entrance and comes up at an exit in the same direction, with up to 4 cells in between (Factorio's basic underground belt). The cells in between stay free for other belts!
- Belts must not mix: every belt connects its own start to its own end.
- Some 2x2 detours are forbidden, as are underground belts that do not pass under anything. Both can always be replaced by a shorter layout. Such *dominance rules* help the solver.

**Objective:** use as few belt tiles as possible.

In the MiniZinc Challenge, MIP solvers struggled with this problem: on `challenge_12` (12x9 cells, 8 belts), neither Gurobi, nor CPLEX, nor HiGHS found a single feasible layout within 20 minutes, even with 8 cores. CP-SAT proved the optimum in 6 minutes.

## Domain Model

In [ ]:
# Define Problem data structures

Direction = Literal["N", "E", "S", "W"]
STEP = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}  # (row, col) offsets
TURNS = {"N": "WE", "S": "WE", "E": "NS", "W": "NS"}            # allowed turns
AXIS = {"N": "NS", "S": "NS", "E": "EW", "W": "EW"}


class Port(BaseModel, frozen=True):
    """Start or end cell of a belt with its direction."""
    row: int
    col: int
    direction: Direction

    @property
    def cell(self) -> tuple[int, int]:
        return self.row, self.col


class Belt(BaseModel, frozen=True):
    """A belt that must connect its start to its end."""
    id: int
    start: Port
    end: Port


class Factory(BaseModel):
    """Represents the problem instance."""
    name: str = ""
    height: int = Field(ge=1)
    width: int = Field(ge=1)
    tunnel_ranges: list[int] = Field(default=[4], description="max. cells between underground entrance and exit")
    walls: list[tuple[int, int]] = []
    belts: list[Belt]

    @model_validator(mode="after")
    def check_ports(self) -> "Factory":
        ports = [p.cell for b in self.belts for p in (b.start, b.end)]
        if len(set(ports)) != len(ports):
            raise ValueError("Two ports share a cell")
        for r, c in ports:
            if not (0 <= r < self.height and 0 <= c < self.width) or (r, c) in self.walls:
                raise ValueError(f"Port ({r}, {c}) is outside the grid or on a wall")
        return self

In [ ]:
# Define Solution and helper functions
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!    You can skip this Code      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


class Tile(BaseModel, frozen=True):
    """A belt piece on the grid."""
    row: int
    col: int
    kind: Literal["belt", "in", "out"]   # surface belt, underground entrance, underground exit
    direction: Direction
    belt: int
    target: tuple[int, int] | None = None  # exit cell of an underground entrance


class FactoryLayout(BaseModel):
    """A feasible layout: every belt runs from its start to its end."""
    factory: Factory
    tiles: list[Tile]

    @property
    def num_tiles(self) -> int:
        return len(self.tiles)

    @model_validator(mode="after")
    def check_belts(self) -> "FactoryLayout":
        at = {(t.row, t.col): t for t in self.tiles}
        if len(at) != len(self.tiles):
            raise ValueError("Two tiles share a cell")
        visited = set()
        for belt in self.factory.belts:
            cell, previous = belt.start.cell, None
            for _ in range(len(self.tiles)):
                tile = at.get(cell)
                if tile is None or tile.belt != belt.id:
                    raise ValueError(f"Belt {belt.id} is broken at {cell}")
                if previous is not None:
                    if previous.kind == "in" and tile.kind != "out":
                        raise ValueError(f"Underground belt at {cell} has no exit")
                    if previous.kind != "in" and (tile.kind == "out" or tile.direction == opposite(previous.direction)):
                        raise ValueError(f"Invalid transition into {cell}")
                visited.add(cell)
                if cell == belt.end.cell:
                    break
                dr, dc = STEP[tile.direction]
                cell, previous = (tile.target if tile.kind == "in" else (cell[0] + dr, cell[1] + dc)), tile
            else:
                raise ValueError(f"Belt {belt.id} never reaches its end")
        if visited != set(at):
            raise ValueError("Some tiles do not belong to any belt")
        return self


def opposite(direction: str) -> str:
    return {"N": "S", "S": "N", "E": "W", "W": "E"}[direction]


def plot_factory(factory: Factory, layout: FactoryLayout | None = None, title: str | None = None):
    """Plot the grid, the ports and, if given, the belt layout."""
    H, W = factory.height, factory.width
    colors = plt.cm.tab10.colors
    arrow = {"N": (0, 0.3), "E": (0.3, 0), "S": (0, -0.3), "W": (-0.3, 0)}
    center = lambda r, c: (c + 0.5, H - 0.5 - r)
    fig, ax = plt.subplots(figsize=(0.55 * W + 1, 0.55 * H + 1))

    for r in range(H):
        for c in range(W):
            ax.add_patch(mpatches.Rectangle((c, H - 1 - r), 1, 1, facecolor="whitesmoke", edgecolor="lightgrey"))
    for r, c in factory.walls:
        ax.add_patch(mpatches.Rectangle((c, H - 1 - r), 1, 1, facecolor="dimgrey"))
    for belt in factory.belts:
        for port in (belt.start, belt.end):
            ax.add_patch(mpatches.Rectangle((port.col + 0.04, H - 1 - port.row + 0.04), 0.92, 0.92, fill=False,
                                            edgecolor=colors[belt.id % 10], linewidth=2.5))
    for tile in layout.tiles if layout else []:
        color = colors[tile.belt % 10]
        x, y = center(tile.row, tile.col)
        dx, dy = arrow[tile.direction]
        underground = tile.kind != "belt"
        ax.add_patch(mpatches.Rectangle((tile.col + 0.12, H - 1 - tile.row + 0.12), 0.76, 0.76, facecolor=color,
                                        alpha=0.9 if underground else 0.3))
        ax.annotate("", xy=(x + dx, y + dy), xytext=(x - dx, y - dy),
                    arrowprops=dict(arrowstyle="-|>", color="white" if underground else color, lw=2))
        if tile.kind == "in":
            (x2, y2) = center(*tile.target)
            ax.plot([x, x2], [y, y2], linestyle=":", color=color, linewidth=3, zorder=3)

    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect("equal")
    ax.axis("off")
    if layout:
        ax.legend(handles=[mpatches.Patch(facecolor="grey", alpha=0.3, label="surface belt"),
                           mpatches.Patch(facecolor="grey", alpha=0.9, label="underground entrance / exit")],
                  loc="upper center", bbox_to_anchor=(0.5, 0), ncol=2, frameon=False)
    ax.set_title(title or (f"{factory.name}: {layout.num_tiles} tiles" if layout else factory.name))
    plt.show()

## Baby Example

Two belts on a 4x6 grid that have to cross each other.

In [ ]:
baby_factory = Factory(
    name="baby",
    height=4,
    width=6,
    belts=[
        Belt(id=0, start=Port(row=0, col=0, direction="E"), end=Port(row=3, col=5, direction="E")),
        Belt(id=1, start=Port(row=3, col=0, direction="E"), end=Port(row=0, col=5, direction="E")),
    ],
)
plot_factory(baby_factory)

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

This is the question for our second breakout session. Think about it before you continue:

- How can you make sure that every belt is one connected path from its start to its end?
- How do you make sure that belts never mix?
- How would you model the underground belts?

When you feel like you engaged enough with this matter, feel free to continue reading!

</div>

## Modeling Idea: One Big Circuit

Think of the grid as a graph: every cell is a node, and every possible belt piece is an arc from its cell to the cell it feeds into:
- a **surface belt** or an **underground exit** in direction $d$: arc to the neighbor in direction $d$,
- an **underground entrance** in direction $d$: arc to a cell 2 to 5 cells further in direction $d$ (where the exit will be).

Now the trick: we connect the end of belt $a$ to the start of belt $a+1$ (and the last end to the first start). All belts together then form **one big circuit**! Every unused cell gets a self-loop. This is exactly what `add_circuit` supports: a circuit through a subset of the nodes, where all other nodes have their self-loop.

| Variable | Description |
|----------|-------------|
| $move_{u,m}$ | cell $u$ holds belt piece $m$, i.e. the arc from $u$ to $target(m)$ is used |
| $empty_u$ | cell $u$ is empty (self-loop) |
| $owner_{u,a}$ | cell $u$ carries belt $a$ |

Constraints:
1. `add_circuit` over all arcs: one big loop through all belts.
2. Valid transitions: $move_{u,m} \Rightarrow$ the piece at $target(m)$ fits (straight, turn or dive; an entrance is followed by its exit).
3. **No mixing:** $move_{u,m} \Rightarrow owner_{u,a} = owner_{target(m),a}$ for all belts $a$, and every start and end is owned by its belt.
4. Underground rules and forbidden 2x2 detours.

**Objective:** minimize the number of non-empty cells.

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

The last gap of this notebook!

The model below does not yet prevent belts from mixing: belt 0 could end up at the end of belt 1.

**What you need to do:**
- Every cell carries at most one belt: the `owner` literals of a cell plus its `empty` literal sum up to exactly one (`model.add_exactly_one(...)`). Start and end cells always carry a belt.
- If a move from `cell` to `mv.target` is used, both cells have the same owner (for every belt).

When you are done, run the next cells. For `challenge_01` you should see:
>```
>challenge_01: OPTIMAL after 3.31s, 43 tiles (bound 43)
>```
If you get fewer than 43 tiles, your belts probably still mix!

</div>

In [ ]:
# Possible pieces on a cell: surface belt, underground entrance ("in") or underground exit ("out")
class Move(NamedTuple):
    kind: str
    direction: str
    tier: int  # max. cells in between for underground belts
    target: tuple[int, int]
    literal: cp_model.IntVar


# 2x2 windows (top-left, top-right, bottom-left, bottom-right) that are never needed: detours and loops
FORBIDDEN_SHAPES = ["SEEN", "NSNW", "SNEN", "WSNW", "SWWN", "ESNS", "SWSN", "ESNE", "SWES",
                    "EENW", "SWEE", "ENNW", "NWEN", "ESWW", "WWEN", "ESSW", "ESNW", "SWEN"]


def create_factory_model(factory: Factory):
    """Create the CP-SAT model for routing all belts of a factory."""
    model = cp_model.CpModel()
    H, W = factory.height, factory.width
    walls = set(map(tuple, factory.walls))
    starts = {b.start.cell: b for b in factory.belts}
    ends = {b.end.cell: b for b in factory.belts}
    cells = [(r, c) for r in range(H) for c in range(W)]
    index = {cell: i for i, cell in enumerate(cells)}
    true = model.new_constant(1)

    def free(cell):
        return 0 <= cell[0] < H and 0 <= cell[1] < W and cell not in walls

    # ---- Variables: one literal per possible piece on a cell --------------------------
    moves = {cell: [] for cell in cells}
    empty = {}
    for cell in cells:
        if cell in walls or cell in ends:  # end cells are fixed; they connect to the next start
            continue
        if cell not in starts:
            empty[cell] = model.new_bool_var(f"empty{cell}")
        for d, (dr, dc) in STEP.items():
            if cell in starts and d != starts[cell].start.direction:
                continue
            neighbor = (cell[0] + dr, cell[1] + dc)
            if free(neighbor) and neighbor not in starts:
                moves[cell].append(Move("belt", d, 0, neighbor, model.new_bool_var("")))
                if cell not in starts:
                    for tier in factory.tunnel_ranges:
                        moves[cell].append(Move("out", d, tier, neighbor, model.new_bool_var("")))
            if cell in starts:
                continue
            for tier in factory.tunnel_ranges:
                for k in range(2, tier + 2):
                    target = (cell[0] + k * dr, cell[1] + k * dc)
                    if free(target) and target not in starts and target not in ends:
                        moves[cell].append(Move("in", d, tier, target, model.new_bool_var("")))

    def pieces(cell, kind, d, tier=None):
        """All literals that put a piece of this kind and direction on a cell."""
        if cell in ends:
            return [true] if kind == "belt" and d == ends[cell].end.direction else []
        return [mv.literal for mv in moves[cell]
                if mv.kind == kind and mv.direction == d and (tier is None or mv.tier == tier)]

    # ---- 1. One big circuit: end of belt a -> start of belt a+1 --------------------------
    arcs = [(index[cell], index[cell], true) for cell in walls]
    arcs += [(index[cell], index[cell], lit) for cell, lit in empty.items()]
    parallel = {}  # a surface belt and an underground exit pointing at the same cell share one arc
    for cell in cells:
        for mv in moves[cell]:
            parallel.setdefault((cell, mv.target), []).append(mv.literal)
    for (tail, head), lits in parallel.items():
        arc = lits[0] if len(lits) == 1 else model.new_bool_var("")
        if len(lits) > 1:
            model.add(arc == sum(lits))
        arcs.append((index[tail], index[head], arc))
    belts = factory.belts
    for a, belt in enumerate(belts):
        arcs.append((index[belt.end.cell], index[belts[(a + 1) % len(belts)].start.cell], true))
    model.add_circuit(arcs)

    for cell in starts:  # a start holds a surface belt in its direction
        model.add_exactly_one(mv.literal for mv in moves[cell])

    # ---- 2. Valid transitions ----------------------------------------------------------------
    for cell in cells:
        for mv in moves[cell]:
            if mv.kind == "in":  # an entrance is followed by an exit of the same tier
                allowed = pieces(mv.target, "out", mv.direction, mv.tier)
            else:                # go straight, turn left or right, or dive underground
                allowed = pieces(mv.target, "belt", mv.direction)
                for turn in TURNS[mv.direction]:
                    allowed += pieces(mv.target, "belt", turn)
                allowed += pieces(mv.target, "in", mv.direction)
            if allowed:
                model.add_bool_or(allowed).only_enforce_if(mv.literal)
            else:
                model.add(mv.literal == 0)

    # ---- 3. Belts must not mix ----------------------------------------------------------------
    owner = {cell: [model.new_bool_var(f"owner{cell}_{b.id}") for b in belts]
             for cell in cells if cell not in walls}
    for belt in belts:
        model.add(owner[belt.start.cell][belt.id] == 1)
        model.add(owner[belt.end.cell][belt.id] == 1)
    # GAP HERE
    # 1. every cell is either empty or carries exactly one belt
    # 2. if a move from cell to mv.target is used, both cells have the same owner

    # ---- 4. Underground rules ------------------------------------------------------------------
    for (r, c), cell_moves in moves.items():
        for mv in cell_moves:
            dr, dc = STEP[mv.direction]
            if mv.kind == "in":
                length = abs(mv.target[0] - r) + abs(mv.target[1] - c)
                for j in range(1, length):
                    below = (r + j * dr, c + j * dc)
                    # no other underground piece of the same tier on the same axis in between
                    for other in moves.get(below, []):
                        if other.kind != "belt" and other.tier == mv.tier and AXIS[other.direction] == AXIS[mv.direction]:
                            model.add_implication(mv.literal, ~other.literal)
                if (r + dr, c + dc) in empty:  # an entrance must pass under something
                    model.add_implication(mv.literal, ~empty[(r + dr, c + dc)])
            if mv.kind == "out" and (r - dr, c - dc) in empty:  # so must an exit
                model.add_implication(mv.literal, ~empty[(r - dr, c - dc)])

    # ---- 4. Forbidden 2x2 detours --------------------------------------------------------------
    for r in range(H - 1):
        for c in range(W - 1):
            window = [(r, c), (r, c + 1), (r + 1, c), (r + 1, c + 1)]
            for shape in FORBIDDEN_SHAPES:
                lits = [pieces(cell, "belt", d) for cell, d in zip(window, shape)]
                if all(lits):
                    model.add_bool_or([~lit[0] for lit in lits])

    # ---- Objective: as few belt tiles as possible ----------------------------------------------
    model.minimize(len(starts) + len(ends) + sum(~lit for lit in empty.values()))
    return model, moves, owner

In [ ]:
def solve_factory(factory: Factory, time_limit: float = 60, verbose: bool = False) -> FactoryLayout | None:
    """Solve the routing problem and return a validated layout (or None)."""
    model, moves, owner = create_factory_model(factory)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.log_search_progress = verbose
    status = solver.solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"{factory.name}: {solver.status_name(status)} after {solver.wall_time:.2f}s")
        return None
    print(f"{factory.name}: {solver.status_name(status)} after {solver.wall_time:.2f}s, "
          f"{solver.objective_value:.0f} tiles (bound {solver.best_objective_bound:.0f})")

    def belt_of(cell):
        return next((b for b, lit in enumerate(owner[cell]) if solver.boolean_value(lit)), -1)

    tiles = [Tile(row=b.end.row, col=b.end.col, kind="belt", direction=b.end.direction, belt=b.id)
             for b in factory.belts]
    for cell, cell_moves in moves.items():
        for mv in cell_moves:
            if solver.boolean_value(mv.literal):
                tiles.append(Tile(row=cell[0], col=cell[1], kind=mv.kind, direction=mv.direction, belt=belt_of(cell),
                                  target=mv.target if mv.kind == "in" else None))
    return FactoryLayout(factory=factory, tiles=tiles)

# This is where you test your homework! Run the next cells.

In [ ]:
baby_layout = solve_factory(baby_factory)
plot_factory(baby_factory, baby_layout)

In [ ]:
factory_01 = Factory(**load_instance("orthorio", "challenge_01"))
layout_01 = solve_factory(factory_01)
plot_factory(factory_01, layout_01)

A slightly bigger instance. Proving optimality takes about a minute on two cores, so be patient.

In [ ]:
factory_03 = Factory(**load_instance("orthorio", "challenge_03"))
layout_03 = solve_factory(factory_03, time_limit=120)
plot_factory(factory_03, layout_03)

### Takeaways

- **The circuit trick:** chaining all belts into one big loop turns multi-belt routing into a single `add_circuit` constraint. Self-loops make nodes optional.
- **Booleans propagate well.** A first version of this model used one integer label per cell (`label[u] == label[v]`). The Boolean `owner` literals made it 2 to 3 times faster.
- **Dominance rules** (forbidden detours, useless underground belts) cut the search space without losing optimal solutions.

---
# Key Takeaways

## When to use CP-SAT, when to use MIP?

| CP-SAT shines | MIP shines |
|---------------|------------|
| Scheduling with disjunctions: intervals, no-overlap, calendars | Strong LP relaxations, e.g. flows, network design |
| Logical and combinatorial constraints, pure feasibility | Continuous variables, huge linear models |
| Routing with side constraints (`add_circuit`) | Facility location, classic TSP formulations |
| Small to medium integer domains | Very large numbers, fractional data |

In the MiniZinc Challenge 2026, MIP solvers won on problems like warehouse location and TSP-like problems. **Know both tools.** CP-SAT itself also contains an LP relaxation, so it brings a bit of both worlds.

## Best practices for scalable CP-SAT models

1. **Tight domains** for every integer variable (e.g. start times within the task's time window).
2. **Enforcement literals** (`only_enforce_if`) instead of Big-M constraints.
3. **Global constraints** (`add_automaton`, `add_no_overlap`, `add_circuit`, ...) instead of hand-made decompositions.
4. **Break symmetries** and add **dominance rules**.
5. **Booleans** over integer "labels" where possible.
6. **Use all cores** (`num_workers`, by default all available) and read the log (`log_search_progress = True`).
7. Integers only: **scale** fractional data.

## Further Resources

- [The CP-SAT Primer](https://d-krupke.github.io/cpsat-primer/) by Dominik Krupke: the best guide to CP-SAT
- [OR-Tools documentation](https://developers.google.com/optimization/cp)
- [CP-SAT masterclass at CPAIOR 2020](https://www.youtube.com/watch?v=lmy1ddn4cyw) by Laurent Perron and Frédéric Didier: how CP-SAT works inside
- [MiniZinc Challenge 2026 results](https://www.minizinc.org/challenge/2026/results/)

<div style="background-color: #2195f33c; border: 2px solid #2196F3; border-radius: 8px; padding: 16px; margin: 16px 0;">

# **💡 BEFORE YOU CONTINUE**

Well, we are at the end of the notebook! Well done!

Here are some questions that we will discuss together in the hands-on live session on [DATE]:

1. Why do MIP solvers struggle with nonograms, even though Model 1 is completely linear?
2. Set `symmetry_breaking=False` for `instance_19`. What happens to the bound in the solver log (`verbose=True`)?
3. Try the bigger factory `challenge_12` with a time limit of 60 seconds. How good is your solution? Can you think of additional constraints that would help the solver?

Thank you for sticking with me to the end!

</div>

---
# Thank You!

Questions?

**Let's connect on LinkedIn:** [Jens-Peter Joost](https://www.linkedin.com/in/jens-peter-joost/)

**More O4A hands-on sessions:** [Optimization4All on YouTube](https://www.youtube.com/@Optimization4All/playlists)